## Imports

### Install packages

In [3]:
if False:
    !sudo /bin/bash -c "(source /venv/bin/activate; pip install --quiet jupyterlab-vim)"
    !jupyter labextension enable

### Import modules

In [ ]:
%load_ext autoreload
%autoreload 2

import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from ipywidgets import interact, FloatSlider, IntSlider, widgets
from IPython.display import display, Markdown

# Set plotting style.
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
import msml610_utils as ut
ut.config_notebook()
    
# Initialize logger.
logging.basicConfig(level=logging.INFO)
_LOG = logging.getLogger(__name__)

<a name='entropy'></a>
# 1. Entropy and Uncertainty

## Definition

**Entropy** $H(X)$ of a discrete random variable $X$ is defined as:

$$H(X) = -\sum_x p(x) \log_2 p(x)$$

## Intuition

- Entropy quantifies the average level of **information**, **surprise**, or **uncertainty** inherent in the variable's possible outcomes
- High entropy = more unpredictability
- Low entropy = more certainty
- Unit: bits (when using $\log_2$)

## Examples

- **Fair coin**: Two equally likely outcomes → maximum uncertainty, $H = 1$ bit
- **Biased coin**: If heads occurs 90% of the time → less uncertainty, $H < 1$ bit

In [ ]:
def calculate_entropy(probabilities):
    """
    Calculate Shannon entropy for a discrete probability distribution.
    
    :param probabilities: Array of probabilities (must sum to 1)
    :return: Entropy in bits
    """
    # Filter out zero probabilities to avoid log(0).
    probabilities = np.array(probabilities)
    probabilities = probabilities[probabilities > 0]
    # Calculate entropy using log base 2.
    entropy = -np.sum(probabilities * np.log2(probabilities))
    return entropy


def binary_entropy(p):
    """
    Calculate entropy of a binary random variable.
    
    :param p: Probability of outcome 1
    :return: Binary entropy H(p)
    """
    if p == 0 or p == 1:
        return 0
    return -p * np.log2(p) - (1 - p) * np.log2(1 - p)


# Test with fair coin.
fair_coin = [0.5, 0.5]
print(f"Fair coin entropy: {calculate_entropy(fair_coin):.4f} bits")

# Test with biased coin.
biased_coin = [0.9, 0.1]
print(f"Biased coin (90-10) entropy: {calculate_entropy(biased_coin):.4f} bits")

# Test with certain outcome.
certain = [1.0, 0.0]
print(f"Certain outcome entropy: {calculate_entropy(certain):.4f} bits")

## Interactive Visualization: Binary Entropy

Use the slider below to adjust the probability $p$ of a binary random variable and observe how entropy changes.

In [ ]:
def plot_binary_entropy_interactive(p=0.5):
    """
    Plot binary entropy function with current value highlighted.
    
    :param p: Probability of outcome 1 (slider controlled)
    """
    # Create array of probabilities.
    p_values = np.linspace(0.001, 0.999, 1000)
    entropy_values = [binary_entropy(p_val) for p_val in p_values]
    # Create figure with two subplots.
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    # Plot 1: Binary entropy function.
    ax1.plot(p_values, entropy_values, 'b-', linewidth=2, label='H(p)')
    ax1.axvline(p, color='red', linestyle='--', linewidth=2, label=f'p = {p:.2f}')
    ax1.axhline(binary_entropy(p), color='red', linestyle='--', linewidth=1, alpha=0.5)
    ax1.scatter([p], [binary_entropy(p)], color='red', s=100, zorder=5)
    ax1.set_xlabel('Probability p', fontsize=12)
    ax1.set_ylabel('Entropy H(p) [bits]', fontsize=12)
    ax1.set_title('Binary Entropy Function', fontsize=14)
    ax1.grid(True, alpha=0.3)
    ax1.legend(fontsize=11)
    ax1.set_ylim([-0.05, 1.1])
    # Plot 2: Probability distribution.
    outcomes = ['Outcome 0', 'Outcome 1']
    probs = [1-p, p]
    colors = ['skyblue', 'coral']
    ax2.bar(outcomes, probs, color=colors, alpha=0.7, edgecolor='black')
    ax2.set_ylabel('Probability', fontsize=12)
    ax2.set_title(f'Probability Distribution\nEntropy = {binary_entropy(p):.4f} bits', fontsize=14)
    ax2.set_ylim([0, 1.1])
    ax2.grid(True, alpha=0.3, axis='y')
    # Add probability values on bars.
    for i, (outcome, prob) in enumerate(zip(outcomes, probs)):
        ax2.text(i, prob + 0.02, f'{prob:.2f}', ha='center', fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()
    # Print information content.
    if p > 0 and p < 1:
        info_0 = -np.log2(1-p)
        info_1 = -np.log2(p)
        print(f"Information content of Outcome 0: {info_0:.4f} bits")
        print(f"Information content of Outcome 1: {info_1:.4f} bits")
        print(f"Expected information (Entropy): {binary_entropy(p):.4f} bits")
    

# Create interactive widget.
interact(plot_binary_entropy_interactive, 
         p=FloatSlider(min=0.01, max=0.99, step=0.01, value=0.5, 
                      description='Probability p:', style={'description_width': 'initial'}));

<a name='joint-conditional'></a>
# 2. Joint and Conditional Entropy

## Joint Entropy

**Joint entropy** $H(X, Y)$ of two variables $X$ and $Y$:

$$H(X, Y) = -\sum_{x,y} p(x,y) \log_2 p(x,y)$$

- Describes the information needed for the joint distribution of $X$ and $Y$
- For independent variables: $H(X, Y) = H(X) + H(Y)$

## Conditional Entropy

**Conditional entropy** $H(Y|X)$ measures uncertainty in $Y$ after observing $X$:

$$H(Y|X) = -\sum_{x,y} p(x,y) \log_2 p(y|x) = \sum_x p(x) H(Y|X=x)$$

**Properties:**
- Low $H(Y|X)$ implies $X$ has strong predictive power for $Y$
- If $Y = X$, then $H(Y|X) = 0$ (no uncertainty)
- If $X$ and $Y$ are independent, then $H(Y|X) = H(Y)$

## Chain Rule for Entropy

$$H(X, Y) = H(X) + H(Y|X) = H(Y) + H(X|Y)$$

In [ ]:
def calculate_joint_entropy(joint_prob):
    """
    Calculate joint entropy H(X,Y) from joint probability distribution.
    
    :param joint_prob: 2D array of joint probabilities p(x,y)
    :return: Joint entropy in bits
    """
    joint_prob = np.array(joint_prob)
    # Filter out zero probabilities.
    joint_prob_flat = joint_prob.flatten()
    joint_prob_flat = joint_prob_flat[joint_prob_flat > 0]
    return -np.sum(joint_prob_flat * np.log2(joint_prob_flat))


def calculate_conditional_entropy(joint_prob):
    """
    Calculate conditional entropy H(Y|X) from joint probability distribution.
    
    :param joint_prob: 2D array of joint probabilities p(x,y)
    :return: Conditional entropy H(Y|X) in bits
    """
    joint_prob = np.array(joint_prob)
    # Calculate marginal p(x).
    p_x = joint_prob.sum(axis=1)
    conditional_entropy = 0
    for i, px in enumerate(p_x):
        if px > 0:
            # Calculate p(y|x) for this x.
            p_y_given_x = joint_prob[i, :] / px
            # Calculate H(Y|X=x).
            p_y_given_x = p_y_given_x[p_y_given_x > 0]
            h_y_given_x = -np.sum(p_y_given_x * np.log2(p_y_given_x))
            # Weight by p(x).
            conditional_entropy += px * h_y_given_x
    return conditional_entropy


# Example: Weather and Activity.
# X = Weather (0: Sunny, 1: Rainy)
# Y = Activity (0: Park, 1: Cinema)
joint_prob = np.array([
    [0.35, 0.15],  # Sunny: 35% park, 15% cinema
    [0.10, 0.40]   # Rainy: 10% park, 40% cinema
])

print("Joint Probability Distribution:")
print(pd.DataFrame(joint_prob, 
                   index=['Sunny', 'Rainy'], 
                   columns=['Park', 'Cinema']))
print()

# Calculate marginals.
p_weather = joint_prob.sum(axis=1)
p_activity = joint_prob.sum(axis=0)

h_weather = calculate_entropy(p_weather)
h_activity = calculate_entropy(p_activity)
h_joint = calculate_joint_entropy(joint_prob)
h_activity_given_weather = calculate_conditional_entropy(joint_prob)

print(f"H(Weather) = {h_weather:.4f} bits")
print(f"H(Activity) = {h_activity:.4f} bits")
print(f"H(Weather, Activity) = {h_joint:.4f} bits")
print(f"H(Activity|Weather) = {h_activity_given_weather:.4f} bits")
print()
print(f"Chain rule verification: H(Weather) + H(Activity|Weather) = {h_weather + h_activity_given_weather:.4f} bits")
print(f"Should equal H(Weather, Activity) = {h_joint:.4f} bits")

<a name='mutual-info'></a>
# 3. Mutual Information

## Definition

**Mutual information** $I(X;Y)$ measures how much knowing one variable reduces uncertainty about the other:

$$I(X;Y) = H(X) - H(X|Y) = H(Y) - H(Y|X) = H(X) + H(Y) - H(X,Y)$$

## Intuition

- Measures the shared information between two variables
- Quantifies the reduction in uncertainty about one variable given the other
- Symmetric: $I(X;Y) = I(Y;X)$

## Properties

- Non-negative: $I(X;Y) \geq 0$
- $I(X;Y) = 0$ if and only if $X$ and $Y$ are independent
- $I(X;X) = H(X)$ (self-information equals entropy)
- Higher mutual information indicates stronger relationship

## Applications

- Feature selection in machine learning
- Dimensionality reduction
- Dependency detection in data

In [ ]:
def calculate_mutual_information(joint_prob):
    """
    Calculate mutual information I(X;Y) from joint probability distribution.
    
    :param joint_prob: 2D array of joint probabilities p(x,y)
    :return: Mutual information in bits
    """
    joint_prob = np.array(joint_prob)
    # Calculate marginals.
    p_x = joint_prob.sum(axis=1)
    p_y = joint_prob.sum(axis=0)
    # Calculate entropies.
    h_x = calculate_entropy(p_x)
    h_y = calculate_entropy(p_y)
    h_xy = calculate_joint_entropy(joint_prob)
    # Mutual information.
    mi = h_x + h_y - h_xy
    return mi


def visualize_information_decomposition(joint_prob):
    """
    Visualize the decomposition of joint entropy into components.
    
    :param joint_prob: 2D array of joint probabilities
    """
    joint_prob = np.array(joint_prob)
    # Calculate all components.
    p_x = joint_prob.sum(axis=1)
    p_y = joint_prob.sum(axis=0)
    h_x = calculate_entropy(p_x)
    h_y = calculate_entropy(p_y)
    h_xy = calculate_joint_entropy(joint_prob)
    h_y_given_x = calculate_conditional_entropy(joint_prob)
    h_x_given_y = calculate_conditional_entropy(joint_prob.T)
    mi = calculate_mutual_information(joint_prob)
    # Create Venn diagram visualization.
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    # Plot 1: Information measures.
    measures = ['H(X)', 'H(Y)', 'H(X,Y)', 'H(Y|X)', 'H(X|Y)', 'I(X;Y)']
    values = [h_x, h_y, h_xy, h_y_given_x, h_x_given_y, mi]
    colors_bars = ['steelblue', 'coral', 'purple', 'lightblue', 'lightsalmon', 'green']
    bars = ax1.bar(measures, values, color=colors_bars, alpha=0.7, edgecolor='black')
    ax1.set_ylabel('Information [bits]', fontsize=12)
    ax1.set_title('Information Measures', fontsize=14)
    ax1.grid(True, alpha=0.3, axis='y')
    ax1.tick_params(axis='x', rotation=45)
    # Add values on bars.
    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    # Plot 2: Relationship diagram.
    ax2.text(0.5, 0.9, 'Information Relationships', ha='center', fontsize=16, fontweight='bold')
    ax2.text(0.5, 0.75, f'H(X, Y) = H(X) + H(Y|X)', ha='center', fontsize=12)
    ax2.text(0.5, 0.68, f'{h_xy:.3f} = {h_x:.3f} + {h_y_given_x:.3f}', ha='center', fontsize=11, color='blue')
    ax2.text(0.5, 0.58, f'I(X;Y) = H(X) + H(Y) - H(X,Y)', ha='center', fontsize=12)
    ax2.text(0.5, 0.51, f'{mi:.3f} = {h_x:.3f} + {h_y:.3f} - {h_xy:.3f}', ha='center', fontsize=11, color='green')
    ax2.text(0.5, 0.41, f'I(X;Y) = H(Y) - H(Y|X)', ha='center', fontsize=12)
    ax2.text(0.5, 0.34, f'{mi:.3f} = {h_y:.3f} - {h_y_given_x:.3f}', ha='center', fontsize=11, color='green')
    ax2.text(0.5, 0.24, 'Interpretation:', ha='center', fontsize=12, fontweight='bold')
    ax2.text(0.5, 0.17, f'Knowing X reduces uncertainty about Y by {mi:.3f} bits', ha='center', fontsize=11)
    ax2.text(0.5, 0.10, f'({(mi/h_y)*100:.1f}% of total uncertainty in Y)', ha='center', fontsize=11, style='italic')
    ax2.axis('off')
    plt.tight_layout()
    plt.show()


# Use the weather-activity example.
print("Example: Weather and Activity Correlation")
print("=" * 50)
visualize_information_decomposition(joint_prob)

# Calculate and display mutual information.
mi = calculate_mutual_information(joint_prob)
print(f"\nMutual Information I(Weather; Activity) = {mi:.4f} bits")
print(f"This means knowing the weather reduces uncertainty about activity by {mi:.4f} bits")

## Interactive Visualization: Correlation and Mutual Information

Adjust the correlation strength to see how it affects mutual information between two variables.

In [ ]:
def create_correlated_joint_distribution(correlation=0.5):
    """
    Create a 2x2 joint distribution with specified correlation.
    
    :param correlation: Correlation strength (0=independent, 1=perfectly correlated)
    :return: 2x2 joint probability matrix
    """
    # Create joint distribution with correlation.
    p11 = 0.25 + correlation * 0.25
    p00 = 0.25 + correlation * 0.25
    p10 = 0.25 - correlation * 0.25
    p01 = 0.25 - correlation * 0.25
    joint_prob = np.array([[p00, p01], [p10, p11]])
    return joint_prob


def plot_mutual_info_interactive(correlation=0.5):
    """
    Interactive plot showing how correlation affects mutual information.
    
    :param correlation: Correlation strength between variables
    """
    joint_prob = create_correlated_joint_distribution(correlation)
    # Calculate metrics.
    mi = calculate_mutual_information(joint_prob)
    p_x = joint_prob.sum(axis=1)
    p_y = joint_prob.sum(axis=0)
    h_x = calculate_entropy(p_x)
    h_y = calculate_entropy(p_y)
    # Create visualization.
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 5))
    # Plot 1: Joint distribution heatmap.
    im = ax1.imshow(joint_prob, cmap='YlOrRd', aspect='auto', vmin=0, vmax=0.5)
    ax1.set_xticks([0, 1])
    ax1.set_yticks([0, 1])
    ax1.set_xticklabels(['Y=0', 'Y=1'])
    ax1.set_yticklabels(['X=0', 'X=1'])
    ax1.set_xlabel('Variable Y', fontsize=12)
    ax1.set_ylabel('Variable X', fontsize=12)
    ax1.set_title(f'Joint Distribution p(X,Y)\nCorrelation = {correlation:.2f}', fontsize=14)
    # Add text annotations.
    for i in range(2):
        for j in range(2):
            text = ax1.text(j, i, f'{joint_prob[i, j]:.3f}',
                          ha="center", va="center", color="black", fontsize=14, fontweight='bold')
    plt.colorbar(im, ax=ax1)
    # Plot 2: Marginal distributions.
    x_pos = np.arange(2)
    width = 0.35
    ax2.bar(x_pos - width/2, p_x, width, label='P(X)', alpha=0.7, color='steelblue')
    ax2.bar(x_pos + width/2, p_y, width, label='P(Y)', alpha=0.7, color='coral')
    ax2.set_xlabel('Outcome', fontsize=12)
    ax2.set_ylabel('Probability', fontsize=12)
    ax2.set_title('Marginal Distributions', fontsize=14)
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(['0', '1'])
    ax2.legend()
    ax2.grid(True, alpha=0.3, axis='y')
    # Plot 3: Information metrics vs correlation.
    correlations = np.linspace(0, 1, 50)
    mis = []
    for corr in correlations:
        jp = create_correlated_joint_distribution(corr)
        mis.append(calculate_mutual_information(jp))
    ax3.plot(correlations, mis, 'b-', linewidth=2, label='I(X;Y)')
    ax3.axvline(correlation, color='red', linestyle='--', linewidth=2, label=f'Current: {correlation:.2f}')
    ax3.scatter([correlation], [mi], color='red', s=100, zorder=5)
    ax3.set_xlabel('Correlation', fontsize=12)
    ax3.set_ylabel('Mutual Information [bits]', fontsize=12)
    ax3.set_title('Mutual Information vs Correlation', fontsize=14)
    ax3.grid(True, alpha=0.3)
    ax3.legend()
    ax3.set_ylim([0, 1])
    plt.tight_layout()
    plt.show()
    # Print metrics.
    print(f"Entropy H(X) = {h_x:.4f} bits")
    print(f"Entropy H(Y) = {h_y:.4f} bits")
    print(f"Mutual Information I(X;Y) = {mi:.4f} bits")
    if h_y > 0:
        print(f"Percentage of Y's entropy explained by X: {(mi/h_y)*100:.2f}%")


# Create interactive widget.
interact(plot_mutual_info_interactive,
         correlation=FloatSlider(min=0.0, max=1.0, step=0.05, value=0.5,
                                description='Correlation:', style={'description_width': 'initial'}));

<a name='kl-cross'></a>
# 4. KL Divergence and Cross-Entropy

## KL Divergence

**Kullback-Leibler (KL) Divergence** $D_{KL}(P \| Q)$ measures how one distribution differs from another:

$$D_{KL}(P \| Q) = \sum_x P(x) \log_2 \frac{P(x)}{Q(x)}$$

**Properties:**
- Not symmetric: $D_{KL}(P \| Q) \neq D_{KL}(Q \| P)$
- Non-negative: $D_{KL}(P \| Q) \geq 0$
- $D_{KL}(P \| Q) = 0$ if and only if $P = Q$
- Not a metric (no triangle inequality)

**Intuition:** Quantifies how much information is lost when $Q$ is used to approximate $P$

## Cross-Entropy

**Cross-entropy** $H(P, Q)$ measures the average number of bits needed to encode data from $P$ using code optimized for $Q$:

$$H(P, Q) = -\sum_x P(x) \log_2 Q(x)$$

**Relationship:**
$$H(P, Q) = H(P) + D_{KL}(P \| Q)$$

**Applications:**
- Loss function in classification (logistic regression, neural networks)
- Model evaluation and comparison
- Information compression

In [ ]:
def calculate_kl_divergence(p, q):
    """
    Calculate KL divergence D_KL(P || Q).
    
    :param p: True distribution P
    :param q: Approximating distribution Q
    :return: KL divergence in bits
    """
    p = np.array(p)
    q = np.array(q)
    # Avoid log(0) by filtering.
    mask = (p > 0) & (q > 0)
    kl = np.sum(p[mask] * np.log2(p[mask] / q[mask]))
    return kl


def calculate_cross_entropy(p, q):
    """
    Calculate cross-entropy H(P, Q).
    
    :param p: True distribution P
    :param q: Model distribution Q
    :return: Cross-entropy in bits
    """
    p = np.array(p)
    q = np.array(q)
    # Avoid log(0).
    mask = (p > 0) & (q > 0)
    ce = -np.sum(p[mask] * np.log2(q[mask]))
    return ce


# Example: Classification problem.
# True distribution (ground truth labels).
true_dist = np.array([0.0, 0.0, 1.0, 0.0])  # Class 2 is correct.

# Model predictions (different confidence levels).
model_confident = np.array([0.05, 0.05, 0.85, 0.05])  # Confident and correct.
model_uncertain = np.array([0.25, 0.25, 0.25, 0.25])  # Uncertain (uniform).
model_wrong = np.array([0.05, 0.85, 0.05, 0.05])     # Confident but wrong.

print("Classification Example")
print("=" * 60)
print("True label: Class 2")
print()

for name, model_pred in [("Confident & Correct", model_confident),
                         ("Uncertain", model_uncertain),
                         ("Confident & Wrong", model_wrong)]:
    kl = calculate_kl_divergence(true_dist, model_pred)
    ce = calculate_cross_entropy(true_dist, model_pred)
    h_true = calculate_entropy(true_dist)
    print(f"{name}:")
    print(f"  Model prediction: {model_pred}")
    print(f"  Cross-Entropy: {ce:.4f} bits")
    print(f"  KL Divergence: {kl:.4f} bits")
    print(f"  H(P) + D_KL(P||Q) = {h_true:.4f} + {kl:.4f} = {h_true + kl:.4f} (should equal CE)")
    print()

## Interactive Visualization: KL Divergence and Distribution Comparison

Adjust the parameters to see how KL divergence changes when approximating a distribution.

In [ ]:
def plot_kl_divergence_interactive(p1=0.7, q1=0.5):
    """
    Interactive visualization of KL divergence between two binary distributions.
    
    :param p1: Probability for true distribution P
    :param q1: Probability for approximating distribution Q
    """
    # Create distributions.
    p = np.array([1-p1, p1])
    q = np.array([1-q1, q1])
    # Calculate metrics.
    kl_pq = calculate_kl_divergence(p, q)
    kl_qp = calculate_kl_divergence(q, p)
    ce_pq = calculate_cross_entropy(p, q)
    h_p = calculate_entropy(p)
    # Create visualization.
    fig = plt.figure(figsize=(16, 10))
    gs = fig.add_gridspec(2, 2, hspace=0.3, wspace=0.3)
    # Plot 1: Distributions comparison.
    ax1 = fig.add_subplot(gs[0, 0])
    x = np.arange(2)
    width = 0.35
    bars1 = ax1.bar(x - width/2, p, width, label='P (True)', alpha=0.7, color='steelblue', edgecolor='black')
    bars2 = ax1.bar(x + width/2, q, width, label='Q (Approximation)', alpha=0.7, color='coral', edgecolor='black')
    ax1.set_xlabel('Outcome', fontsize=12)
    ax1.set_ylabel('Probability', fontsize=12)
    ax1.set_title('Distribution Comparison', fontsize=14, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(['0', '1'])
    ax1.legend(fontsize=11)
    ax1.set_ylim([0, 1.1])
    ax1.grid(True, alpha=0.3, axis='y')
    # Add value labels.
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                    f'{height:.2f}', ha='center', va='bottom', fontsize=10)
    # Plot 2: KL divergence metrics.
    ax2 = fig.add_subplot(gs[0, 1])
    metrics = ['H(P)', 'H(P,Q)', 'D_KL(P||Q)', 'D_KL(Q||P)']
    values = [h_p, ce_pq, kl_pq, kl_qp]
    colors_m = ['steelblue', 'purple', 'red', 'orange']
    bars = ax2.bar(metrics, values, color=colors_m, alpha=0.7, edgecolor='black')
    ax2.set_ylabel('Information [bits]', fontsize=12)
    ax2.set_title('Information Metrics', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='y')
    ax2.tick_params(axis='x', rotation=20)
    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    # Plot 3: KL divergence heatmap.
    ax3 = fig.add_subplot(gs[1, :])
    p_range = np.linspace(0.05, 0.95, 30)
    q_range = np.linspace(0.05, 0.95, 30)
    kl_matrix = np.zeros((len(p_range), len(q_range)))
    for i, p_val in enumerate(p_range):
        for j, q_val in enumerate(q_range):
            p_dist = np.array([1-p_val, p_val])
            q_dist = np.array([1-q_val, q_val])
            kl_matrix[i, j] = calculate_kl_divergence(p_dist, q_dist)
    im = ax3.contourf(q_range, p_range, kl_matrix, levels=20, cmap='RdYlBu_r')
    ax3.scatter([q1], [p1], color='red', s=200, marker='*', edgecolor='black', linewidth=2, 
               label=f'Current: D_KL={kl_pq:.3f}', zorder=5)
    ax3.set_xlabel('Q (Approximation probability for outcome 1)', fontsize=12)
    ax3.set_ylabel('P (True probability for outcome 1)', fontsize=12)
    ax3.set_title('KL Divergence D_KL(P||Q) Heatmap', fontsize=14, fontweight='bold')
    ax3.legend(fontsize=11, loc='upper left')
    cbar = plt.colorbar(im, ax=ax3)
    cbar.set_label('KL Divergence [bits]', fontsize=11)
    # Add diagonal line (where P=Q).
    ax3.plot([0, 1], [0, 1], 'k--', linewidth=2, alpha=0.5, label='P=Q (KL=0)')
    plt.tight_layout()
    plt.show()
    # Print analysis.
    print(f"True distribution P: [{1-p1:.2f}, {p1:.2f}]")
    print(f"Approximation Q:    [{1-q1:.2f}, {q1:.2f}]")
    print()
    print(f"Entropy H(P) = {h_p:.4f} bits")
    print(f"Cross-Entropy H(P,Q) = {ce_pq:.4f} bits")
    print(f"KL Divergence D_KL(P||Q) = {kl_pq:.4f} bits")
    print(f"KL Divergence D_KL(Q||P) = {kl_qp:.4f} bits (note: asymmetric!)")
    print()
    print(f"Verification: H(P) + D_KL(P||Q) = {h_p:.4f} + {kl_pq:.4f} = {h_p+kl_pq:.4f}")
    print(f"Should equal H(P,Q) = {ce_pq:.4f} ✓" if abs(h_p+kl_pq-ce_pq) < 0.001 else f"Should equal H(P,Q) = {ce_pq:.4f} ✗")


# Create interactive widget.
interact(plot_kl_divergence_interactive,
         p1=FloatSlider(min=0.05, max=0.95, step=0.05, value=0.7,
                       description='P(outcome=1):', style={'description_width': 'initial'}),
         q1=FloatSlider(min=0.05, max=0.95, step=0.05, value=0.5,
                       description='Q(outcome=1):', style={'description_width': 'initial'}));

<a name='advanced'></a>
# 5. Advanced Topics

## Data Processing Inequality

**Statement:** Processing data cannot increase information, it can only lose information.

Formally, if $X \to Y \to Z$ forms a Markov chain, then:
$$I(X;Z) \leq I(X;Y)$$

**Intuition:** Information can only be lost through processing, never gained.

**Example:** If $X$ is a raw image and $Y$ is compressed version, no further processing $Z$ will recover more information about $X$ than $Y$ already contains.

## Maximum Entropy Principle

**Principle:** Use the distribution with the largest entropy given the constraints.

**Examples of maximum entropy distributions:**
- No constraints → Uniform distribution
- Positive mean constraint → Exponential distribution  
- Fixed variance → Normal distribution

## Minimum Description Length (MDL)

**Principle:** Select the model that minimizes total description length:

$$MDL(H) = L(H) + L(D | H)$$

where:
- $L(H)$ = length of the model/hypothesis
- $L(D|H)$ = length of data encoded using the model

**Intuition:** Balances model complexity with data fit (Occam's Razor principle)

## Kolmogorov Complexity

**Definition:** The length of the shortest program that outputs a string $x$.

**Examples:**
- String "00000000..." has low complexity (simple loop)
- Random string has high complexity (no pattern, no compression)

**Properties:**
- Not computable (theoretical concept)
- Related to MDL in practice
- Measures algorithmic randomness

In [ ]:
def demonstrate_data_processing_inequality():
    """
    Demonstrate the data processing inequality with a simple example.
    """
    # Create Markov chain X -> Y -> Z.
    # X: Original signal (4 states).
    p_x = np.array([0.4, 0.3, 0.2, 0.1])
    # Y: Compressed version (2 states) - groups (0,1) and (2,3).
    # Z: Further processed (2 states) - adds noise.
    # Transition probabilities.
    # P(Y|X): X states 0,1 -> Y=0 with high prob; X states 2,3 -> Y=1.
    p_y_given_x = np.array([
        [0.9, 0.1],  # X=0 -> mostly Y=0
        [0.85, 0.15],  # X=1 -> mostly Y=0
        [0.1, 0.9],  # X=2 -> mostly Y=1
        [0.05, 0.95]   # X=3 -> mostly Y=1
    ])
    # P(Z|Y): Add noise.
    p_z_given_y = np.array([
        [0.8, 0.2],  # Y=0 -> mostly Z=0
        [0.2, 0.8]   # Y=1 -> mostly Z=1
    ])
    # Calculate joint distributions.
    p_xy = p_x[:, np.newaxis] * p_y_given_x  # Joint P(X,Y)
    p_y = p_xy.sum(axis=0)  # Marginal P(Y)
    p_yz = p_y[:, np.newaxis] * p_z_given_y  # Joint P(Y,Z)
    p_z = p_yz.sum(axis=0)  # Marginal P(Z)
    # Calculate P(X,Z) through Y.
    p_xz = np.zeros((4, 2))
    for i in range(4):
        for k in range(2):
            for j in range(2):
                p_xz[i, k] += p_x[i] * p_y_given_x[i, j] * p_z_given_y[j, k]
    # Calculate mutual informations.
    mi_xy = calculate_mutual_information(p_xy)
    mi_yz = calculate_mutual_information(p_yz)
    mi_xz = calculate_mutual_information(p_xz)
    # Visualize.
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    # Plot 1: Mutual information values.
    stages = ['I(X;Y)', 'I(Y;Z)', 'I(X;Z)']
    mi_values = [mi_xy, mi_yz, mi_xz]
    colors_stages = ['steelblue', 'coral', 'lightgreen']
    bars = ax1.bar(stages, mi_values, color=colors_stages, alpha=0.7, edgecolor='black', linewidth=2)
    ax1.set_ylabel('Mutual Information [bits]', fontsize=12)
    ax1.set_title('Data Processing Inequality\nX → Y → Z', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3, axis='y')
    # Add values and inequality annotations.
    for bar, val in zip(bars, mi_values):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax1.axhline(mi_xy, color='steelblue', linestyle='--', alpha=0.5, linewidth=2)
    ax1.text(1.5, mi_xy + 0.03, f'I(X;Y) bound', fontsize=10, style='italic')
    # Plot 2: Information flow diagram.
    ax2.text(0.5, 0.9, 'Data Processing Inequality', ha='center', fontsize=16, fontweight='bold')
    ax2.text(0.5, 0.75, 'Markov Chain: X → Y → Z', ha='center', fontsize=13, style='italic')
    ax2.text(0.5, 0.65, f'I(X;Y) = {mi_xy:.4f} bits', ha='center', fontsize=12, color='steelblue')
    ax2.text(0.5, 0.58, f'I(Y;Z) = {mi_yz:.4f} bits', ha='center', fontsize=12, color='coral')
    ax2.text(0.5, 0.51, f'I(X;Z) = {mi_xz:.4f} bits', ha='center', fontsize=12, color='green')
    ax2.text(0.5, 0.40, 'Data Processing Inequality:', ha='center', fontsize=13, fontweight='bold')
    ax2.text(0.5, 0.33, f'I(X;Z) ≤ I(X;Y)', ha='center', fontsize=12)
    ax2.text(0.5, 0.26, f'{mi_xz:.4f} ≤ {mi_xy:.4f} ✓', ha='center', fontsize=12, 
            color='green' if mi_xz <= mi_xy else 'red', fontweight='bold')
    ax2.text(0.5, 0.16, 'Interpretation:', ha='center', fontsize=12, fontweight='bold')
    ax2.text(0.5, 0.09, f'Information lost from X to Z: {mi_xy - mi_xz:.4f} bits', ha='center', fontsize=11)
    ax2.text(0.5, 0.03, f'({((mi_xy-mi_xz)/mi_xy)*100:.1f}% of original information)', 
            ha='center', fontsize=10, style='italic')
    ax2.axis('off')
    plt.tight_layout()
    plt.show()
    print("Data Processing Inequality Demonstration")
    print("=" * 60)
    print(f"Original signal X has {len(p_x)} states")
    print(f"Compressed Y has 2 states")
    print(f"Processed Z has 2 states")
    print()
    print(f"I(X;Y) = {mi_xy:.4f} bits (information between X and Y)")
    print(f"I(Y;Z) = {mi_yz:.4f} bits (information between Y and Z)")
    print(f"I(X;Z) = {mi_xz:.4f} bits (information between X and Z)")
    print()
    print(f"Data Processing Inequality: I(X;Z) ≤ I(X;Y)")
    print(f"{mi_xz:.4f} ≤ {mi_xy:.4f}: {'✓ Satisfied' if mi_xz <= mi_xy + 1e-6 else '✗ Violated'}")
    print()
    print(f"Information lost: {mi_xy - mi_xz:.4f} bits ({((mi_xy-mi_xz)/mi_xy)*100:.1f}%)")


demonstrate_data_processing_inequality()

# Summary and Key Takeaways

## Information Theory Relationships

The fundamental relationships in information theory:

1. **Entropy relationships:**
   - $H(X, Y) = H(X) + H(Y|X) = H(Y) + H(X|Y)$ (Chain Rule)
   - $H(X|Y) \leq H(X)$ with equality iff $X$ and $Y$ are independent

2. **Mutual information:**
   - $I(X;Y) = H(X) - H(X|Y) = H(Y) - H(Y|X)$
   - $I(X;Y) = H(X) + H(Y) - H(X,Y)$
   - $I(X;Y) \geq 0$ with equality iff $X$ and $Y$ are independent

3. **KL divergence and cross-entropy:**
   - $H(P,Q) = H(P) + D_{KL}(P \| Q)$
   - $D_{KL}(P \| Q) \geq 0$ with equality iff $P = Q$

4. **Data processing inequality:**
   - $I(X;Z) \leq I(X;Y)$ if $X \to Y \to Z$

## Applications in Machine Learning

- **Entropy:** Feature selection, decision tree splitting criteria
- **Mutual Information:** Dependency detection, feature relevance
- **Cross-Entropy:** Loss function for classification
- **KL Divergence:** Variational inference, model comparison
- **MDL:** Model selection, preventing overfitting

# Exercises

Try these exercises to deepen your understanding:

1. **Entropy Exercise:** Calculate the entropy of a 6-sided fair die. What happens if one outcome has probability 0.5 and the others share the remaining probability equally?

2. **Mutual Information Exercise:** Given the joint distribution in the weather-activity example, calculate $I(Weather; Activity)$ manually and verify it matches the computed value.

3. **Cross-Entropy Exercise:** For a 3-class classification problem, compute the cross-entropy loss when:
   - True label: class 1
   - Model prediction: [0.2, 0.6, 0.2]
   
4. **Data Processing Exercise:** Explain why JPEG compression is lossy in terms of the data processing inequality.

5. **Maximum Entropy Exercise:** Show that the uniform distribution maximizes entropy among all distributions with the same number of outcomes.

In [ ]:
# Exercise workspace - use this cell to work on the exercises above.

# Exercise 1: Fair die entropy.
die_probs = np.array([1/6] * 6)
die_entropy = calculate_entropy(die_probs)
print(f"Exercise 1: Fair die entropy = {die_entropy:.4f} bits")

# Your code here for other exercises...